In [2]:
import sys
sys.path.append('../')
from src.load_data import *

In [3]:
testing_data = load_data_from_csv('../data/test_sent_emo.csv')
print(testing_data.head())

Data loaded successfully from ../data/test_sent_emo.csv
   Sr No.                                          Utterance Speaker  \
0       1  Why do all youre coffee mugs have numbers on ...    Mark   
1       2  Oh. Thats so Monica can keep track. That way ...  Rachel   
2       3                                       Y'know what?  Rachel   
3      19                     Come on, Lydia, you can do it.    Joey   
4      20                                              Push!    Joey   

    Emotion Sentiment  Dialogue_ID  Utterance_ID  Season  Episode  \
0  surprise  positive            0             0       3       19   
1     anger  negative            0             1       3       19   
2   neutral   neutral            0             2       3       19   
3   neutral   neutral            1             0       1       23   
4       joy  positive            1             1       1       23   

      StartTime       EndTime  
0  00:14:38,127  00:14:40,378  
1  00:14:40,629  00:14:47,385  


In [4]:
testing_data.drop_duplicates(inplace=True)

In [5]:
testing_data.columns

Index(['Sr No.', 'Utterance', 'Speaker', 'Emotion', 'Sentiment', 'Dialogue_ID',
       'Utterance_ID', 'Season', 'Episode', 'StartTime', 'EndTime'],
      dtype='str')

In [6]:
print(testing_data.head().to_json(orient='records', lines=True))

{"Sr No.":1,"Utterance":"Why do all you\u0092re coffee mugs have numbers on the bottom?","Speaker":"Mark","Emotion":"surprise","Sentiment":"positive","Dialogue_ID":0,"Utterance_ID":0,"Season":3,"Episode":19,"StartTime":"00:14:38,127","EndTime":"00:14:40,378"}
{"Sr No.":2,"Utterance":"Oh. That\u0092s so Monica can keep track. That way if one on them is missing, she can be like, \u0091Where\u0092s number 27?!\u0092","Speaker":"Rachel","Emotion":"anger","Sentiment":"negative","Dialogue_ID":0,"Utterance_ID":1,"Season":3,"Episode":19,"StartTime":"00:14:40,629","EndTime":"00:14:47,385"}
{"Sr No.":3,"Utterance":"Y'know what?","Speaker":"Rachel","Emotion":"neutral","Sentiment":"neutral","Dialogue_ID":0,"Utterance_ID":2,"Season":3,"Episode":19,"StartTime":"00:14:56,353","EndTime":"00:14:57,520"}
{"Sr No.":19,"Utterance":"Come on, Lydia, you can do it.","Speaker":"Joey","Emotion":"neutral","Sentiment":"neutral","Dialogue_ID":1,"Utterance_ID":0,"Season":1,"Episode":23,"StartTime":"0:10:44,769","E

In [7]:
unqiue_emotions = testing_data['Emotion'].unique()
print(unqiue_emotions)

<StringArray>
['surprise', 'anger', 'neutral', 'joy', 'sadness', 'fear', 'disgust']
Length: 7, dtype: str


In [8]:
# Create a unique key like "102_5" (Dialogue_ID + Utterance_ID)
testing_data['Recognition_ID'] = (
    testing_data['Dialogue_ID'].astype(str) + "_" + 
    testing_data['Utterance_ID'].astype(str) + "_" +
    testing_data['Season'].astype(str) + "_" +
    testing_data['Episode'].astype(str)
)

print(testing_data[['Dialogue_ID', 'Utterance_ID', 'Recognition_ID']].head())

   Dialogue_ID  Utterance_ID Recognition_ID
0            0             0       0_0_3_19
1            0             1       0_1_3_19
2            0             2       0_2_3_19
3            1             0       1_0_1_23
4            1             1       1_1_1_23


In [9]:
testing_data.columns

Index(['Sr No.', 'Utterance', 'Speaker', 'Emotion', 'Sentiment', 'Dialogue_ID',
       'Utterance_ID', 'Season', 'Episode', 'StartTime', 'EndTime',
       'Recognition_ID'],
      dtype='str')

In [10]:
testing_data['Recognition_ID'].nunique()

2610

In [11]:
testing_data.shape

(2610, 12)

In [12]:
def meld_context_provider(testing_data, index):
    # 1. Get the IDs for the target row
    target_row = testing_data.iloc[index]
    current_dialogue_id = target_row['Dialogue_ID']
    current_utterance_id = target_row['Utterance_ID']
    recognition_id = target_row['Recognition_ID'] # The unique ID we created
    
    # 2. Filter for previous lines within the same dialogue
    full_dialogue = testing_data[testing_data['Dialogue_ID'] == current_dialogue_id]
    context_rows = full_dialogue[full_dialogue['Utterance_ID'] < current_utterance_id]
    
    # 3. Grab the last 3 utterances for context
    last_context = context_rows.tail(3)
    
    # 4. Format the context dictionary
    context = {
        "recognition_id": recognition_id,  # Essential for your logging
        "history": [],
        "target": {
            "speaker": target_row['Speaker'],
            "utterance": target_row['Utterance']
        }
    }
    
    if last_context.empty:
        context["history"] = ["No previous dialogue available (Start of Conversation)."]
    else:
        # We format history as strings like "Speaker: Utterance" 
        # because LLMs process this 'dialogue format' better.
        context["history"] = [
            f"{row['Speaker']}: {row['Utterance']}" 
            for _, row in last_context.iterrows()
        ]
        
    return context

# Test it

context_list = meld_context_provider(testing_data, 1)

print(context_list)

{'recognition_id': '0_1_3_19', 'history': ['Mark: Why do all you\x92re coffee mugs have numbers on the bottom?'], 'target': {'speaker': 'Rachel', 'utterance': 'Oh. That\x92s so Monica can keep track. That way if one on them is missing, she can be like, \x91Where\x92s number 27?!\x92'}}


In [ ]:
import time
import json
import os
from src.agents import run_multi_agent_conversation

def run_batch_processing(dataset, batch_size=10, sleep_time=2):
    total_rows = len(dataset)
    output_file = "../logs/meld_results.jsonl"
    
    # 1. Load existing progress to avoid re-doing work
    processed_ids = set()
    if os.path.exists(output_file):
        with open(output_file, "r") as f:
            for line in f:
                try:
                    processed_ids.add(json.loads(line)['recognition_id'])
                except:
                    continue
    
    print(f"Resuming... {len(processed_ids)} rows already completed.")

    # 2. The Batch Loop
    for i in range(0, total_rows, batch_size):
        batch = dataset.iloc[i : i + batch_size]
        
        print(f"Processing Batch {i} to {i + batch_size}...")
        
        for index, row in batch.iterrows():
            # Check if this specific ID is already done
            rec_id = str(row['Dialogue_ID']) + "_" + str(row['Utterance_ID'])
            if rec_id in processed_ids:
                continue

            try:
                # A. Get Context
                context = meld_context_provider(dataset, index)
                context['recognition_id'] = rec_id # Ensure ID is passed
                
                # B. Run the Agents
                result_text = run_multi_agent_conversation(context)
                
                # C. Save immediately (Append Mode)
                result_json = {
                    "recognition_id": rec_id,
                    "ground_truth": row['Emotion'],
                    "raw_output": result_text
                }
                
                with open(output_file, "a") as f:
                    f.write(json.dumps(result_json) + "\n")
                    
            except Exception as e:
                print(f"Error on {rec_id}: {e}")
                # Optional: Sleep longer if you hit a rate limit
                if "429" in str(e):
                    print("Rate Limit Hit! Sleeping for 60 seconds...")
                    time.sleep(60)
                if "503" in str(e):
                    print("Service Unavailable! Sleeping for 60 seconds...")
                    time.sleep(60) 
                if e is not None:
                    print("Unexpected error! Sleeping for 10 seconds...")
                    time.sleep(10) 
        
        # 3. Batch Sleep (Be nice to the API)
        print(f"Batch {i} complete. Sleeping for {sleep_time} seconds...")
        time.sleep(sleep_time)
    print("All batches processed.")

# Run it
run_batch_processing(testing_data, batch_size=5)

Resuming... 148 rows already completed.
Processing Batch 0 to 5...
Batch 0 complete. Sleeping for 2 seconds...
Processing Batch 5 to 10...
Batch 5 complete. Sleeping for 2 seconds...
Processing Batch 10 to 15...
Batch 10 complete. Sleeping for 2 seconds...
Processing Batch 15 to 20...
Batch 15 complete. Sleeping for 2 seconds...
Processing Batch 20 to 25...
Batch 20 complete. Sleeping for 2 seconds...
Processing Batch 25 to 30...
Batch 25 complete. Sleeping for 2 seconds...
Processing Batch 30 to 35...
Batch 30 complete. Sleeping for 2 seconds...
Processing Batch 35 to 40...
Batch 35 complete. Sleeping for 2 seconds...
Processing Batch 40 to 45...
Batch 40 complete. Sleeping for 2 seconds...
Processing Batch 45 to 50...
Batch 45 complete. Sleeping for 2 seconds...
Processing Batch 50 to 55...
Batch 50 complete. Sleeping for 2 seconds...
Processing Batch 55 to 60...
Batch 55 complete. Sleeping for 2 seconds...
Processing Batch 60 to 65...
Batch 60 complete. Sleeping for 2 seconds...
Pro

In [ ]:
# from src.agents import run_multi_agent_conversation
# for i in range(2):
#     context_list = meld_context_provider(testing_data, i)
#     final_response = run_multi_agent_conversation(context_list)
#     print(f"Final response for index {i}: {final_response}")

Final response for index 0: {
  "recognition_id": "0_0",
  "reasoning": "All expert reports describe a calm, domestic setting where Mark asks a straightforward, curiosity‑driven question without conflict, sarcasm, or strong affect. This consistent low‑arousal, non‑evaluative tone aligns with a neutral emotion.",
  "predicted_emotion": "neutral"
}
Final response for index 1: {
  "recognition_id": "0_1",
  "reasoning": "All three expert reports describe a relaxed, low‑stakes conversation with lighthearted humor and no conflict, and the empathy analysis notes playful exaggeration and moderate arousal. This combination points to a positive, amused mood rather than neutrality or negativity.",
  "predicted_emotion": "joy"
}


In [2]:
import json
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

def get_report(file_path):
    y_true = []
    y_pred = []
    
    with open(file_path, "r") as f:
        for line in f:
            data = json.loads(line)
            # 1. Get Ground Truth
            y_true.append(data['ground_truth'].lower())
            
            # 2. Extract Prediction from the Aggregator's raw string
            try:
                # We search for the JSON block in the 'raw_output'
                import re
                match = re.search(r'\{.*\}', data['raw_output'], re.DOTALL)
                pred_json = json.loads(match.group())
                # Adjust 'predicted_emotion' to whatever key your prompt uses
                y_pred.append(pred_json.get('predicted_emotion', 'neutral').lower())
            except:
                # y_pred.append("neutral") # Fallback for parsing errors
                continue  # Skip this entry if we can't parse it

    # print("### PRELIMINARY EXPERIMENT REPORT (N=500) ###")
    print(classification_report(y_true, y_pred))

# Run it!
print(get_report("../logs/meld_results.jsonl"))

              precision    recall  f1-score   support

       anger       0.41      0.69      0.52        36
     disgust       0.67      0.33      0.44         6
        fear       0.17      0.83      0.28         6
         joy       0.38      0.69      0.49        39
     neutral       0.81      0.29      0.42       119
     sadness       0.40      0.35      0.38        17
    surprise       0.33      0.33      0.33        21

    accuracy                           0.43       244
   macro avg       0.45      0.50      0.41       244
weighted avg       0.59      0.43      0.43       244

None


In [3]:
print(get_report("../logs/meld_results.jsonl"))

              precision    recall  f1-score   support

       anger       0.41      0.69      0.52        36
     disgust       0.67      0.33      0.44         6
        fear       0.17      0.83      0.28         6
         joy       0.38      0.69      0.49        39
     neutral       0.81      0.29      0.42       119
     sadness       0.40      0.35      0.38        17
    surprise       0.33      0.33      0.33        21

    accuracy                           0.43       244
   macro avg       0.45      0.50      0.41       244
weighted avg       0.59      0.43      0.43       244

None
